# AlphaQuest Capital — BTC Macro-Sentiment Signal Pipeline

This notebook provides a step-by-step walkthrough of the AlphaQuest Bitcoin trading strategy. We follow a rigorous quant-finance workflow:

1.  **ETL**: Fetching Bitcoin prices, FRED Macro signals, and Crypto Sentiment.
2.  **Feature Engineering**: Computing technical indicators and composite macro scores.
3.  **Machine Learning**: Training an ensemble of models to predict tomorrow's price direction.
4.  **Backtesting**: Simulating the strategy performance with a 5% stop-loss kill switch.

## 1. Setup and Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully.")

## 2. Configuration & API Keys

We load the FRED API key. In this notebook environment, we'll try to get it from environment variables or a local `.streamlit/secrets.toml` if available.

In [ ]:
FRED_API_KEY = os.environ.get("FRED_API_KEY", "")

if not FRED_API_KEY:
    try:
        import toml
        secrets = toml.load(".streamlit/secrets.toml")
        FRED_API_KEY = secrets.get("FRED_API_KEY", "")
    except:
        pass

START_DATE = "2020-01-01"
END_DATE   = "2024-12-31"

print(f"Pipeline configured for range: {START_DATE} to {END_DATE}")
if not FRED_API_KEY:
    print("⚠️ WARNING: FRED_API_KEY not found. FRED data steps will fail.")

## 3. Data Acquisition (ETL)

We fetch:
- **BTC-USD** (Prices from Yahoo Finance)
- **DFF & SP500** (Macro data from FRED)
- **Fear & Greed** (Sentiment Index)

In [ ]:
import yfinance as yf
from fredapi import Fred
import requests

def get_data():
    # 1. BTC OHLCV
    print("Fetching BTC prices...")
    btc = yf.download("BTC-USD", start=START_DATE, end=END_DATE, progress=False)
    btc.columns = [c.lower() for c in btc.columns]
    
    # 2. FRED Macro
    if FRED_API_KEY:
        print("Fetching FRED Data (DFF, SP500, DGS10)...")
        fred = Fred(api_key=FRED_API_KEY)
        dff = fred.get_series("DFF", observation_start=START_DATE, observation_end=END_DATE)
        sp500 = fred.get_series("SP500", observation_start=START_DATE, observation_end=END_DATE)
        bond = fred.get_series("DGS10", observation_start=START_DATE, observation_end=END_DATE)
        macro = pd.DataFrame({"fed_rate": dff, "sp500": sp500, "bond_yield": bond}).ffill()
    else:
        macro = pd.DataFrame()

    # 3. Fear & Greed
    print("Fetching Fear & Greed Index...")
    res = requests.get("https://api.alternative.me/fng/?limit=0")
    fgi_data = res.json()["data"]
    fgi = pd.DataFrame(fgi_data)
    fgi["date"] = pd.to_datetime(fgi["timestamp"], unit="s")
    fgi.set_index("date", inplace=True)
    fgi["fear_greed"] = fgi["value"].astype(float)
    fgi = fgi[["fear_greed"]].sort_index()

    return btc, macro, fgi

btc_raw, macro_raw, fgi_raw = get_data()
display(btc_raw.head())
display(macro_raw.head())

## 4. Feature Engineering

We calculate technical indicators like **RSI**, **MACD**, and **ADX**. We also create **Composite Features**:
- `trend_score`: Combines EMA crossover and MACD histograms.
- `macro_pressure`: Combines bond yield changes and Fed rate changes.

In [ ]:
def compute_features(df):
    # Technicals
    df['ma5'] = df['close'].rolling(5).mean()
    df['ma20'] = df['close'].rolling(20).mean()
    df['ma_ratio'] = df['ma5'] / df['ma20']
    
    # RSI
    delta = df['close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi'] = 100 - (100 / (1 + gain/loss))
    
    # MACD
    ema12 = df['close'].ewm(span=12).mean()
    ema26 = df['close'].ewm(span=26).mean()
    df['macd_hist'] = ema12 - ema26
    
    # Trend Score (Composite)
    df['trend_score'] = (df['close'] / df['ma20'] - 1) + (df['macd_hist'] / df['close'])
    
    return df.dropna()

df_features = compute_features(btc_raw.copy())
df_features = df_features.join(macro_raw).join(fgi_raw.shift(1)).ffill().dropna()

print("Feature Matrix Created:")
display(df_features.head())

## 5. Model Training & Evaluation

We use a **Random Forest Classifier** as our primary model to predict if tomorrow's return will be positive.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Create Target (Tomorrow's direction)
df_features['target'] = (df_features['close'].shift(-1) > df_features['close']).astype(int)
df_ml = df_features.dropna()

# Split: Train on 2020-2022, Test on 2023-2024
features = ['ma_ratio', 'rsi', 'trend_score', 'fear_greed']
train = df_ml[df_ml.index < '2023-01-01']
test = df_ml[df_ml.index >= '2023-01-01']

rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(train[features], train['target'])

preds = rf.predict(test[features])
print("Model Evaluation Summary:")
print(classification_report(test['target'], preds))

## 6. Trading Strategy Backtest

We simulate a **Long/Cash** strategy:
- If the model predicts 'Up' (Buy), we hold BTC for 1 day.
- If the model predicts 'Down' (Hold), we stay in Cash.
- We apply a **5% daily stop-loss** kill switch.

In [ ]:
test['strategy_ret'] = test['target'].shift(1).fillna(0) * test['close'].pct_change()

# Apply Stop-Loss Kill Switch
test['kill_switch'] = (test['strategy_ret'] < -0.05).astype(int)
test.loc[test['kill_switch'].shift(1) == 1, 'strategy_ret'] = 0

test['cum_strategy'] = (1 + test['strategy_ret']).cumprod()
test['cum_bnh'] = (1 + test['close'].pct_change().fillna(0)).cumprod()

plt.figure(figsize=(14, 7))
plt.plot(test['cum_strategy'], label='AlphaQuest Strategy', color='emerald', lw=2)
plt.plot(test['cum_bnh'], label='Buy & Hold BTC', color='gray', alpha=0.6, ls='--')
plt.title("Strategy Performance Comparison (2023-2024)", fontsize=14)
plt.legend()
plt.show()

## 7. Next-Day Prediction

Finally, we use today's latest data to see what the signal is for tomorrow!

In [ ]:
latest_data = df_ml[features].iloc[[-1]]
prob = rf.predict_proba(latest_data)[0, 1]
signal = "🟢 BUY (Long)" if prob > 0.50 else "⚪ HOLD CASH"

print("="*40)
print(f"Next Day Signal Context: {df_ml.index[-1].date()}")
print(f"Model Confidence (Up %): {prob:.1%}")
print(f"FINAL SIGNAL: {signal}")
print("="*40)